In [ ]:
import pandas as pd
import numpy as np
from astropy.io import fits
import csv
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


data_folder = "./data/Training_Data"
n_inputs = 1000
file_length_limit = 3000

fits_data = []
visibilities = []

with open(f"{data_folder}/uv.txt","r") as file:
    d = csv.reader(file,delimiter=" ")
    uv_data = np.array([[float(y) for y in x if y != ""] for x in d])[:file_length_limit]

uv_mean = np.mean(uv_data, axis=0)
uv_std_dev = np.std(uv_data, axis=0)

for i in tqdm(range(0,n_inputs)):
    fits_file = fits.open(f"{data_folder}/input_image_{i}.fits")
    with open(f"{data_folder}/visibilities_{i}.txt","r") as file:
        d = csv.reader(file,delimiter=" ")
        d = np.array([[float(y) for y in x if y != ""] for x in d][:file_length_limit])
        #uv_distances = np.reshape(np.sqrt(uv_data[:,0]**2 + uv_data[:,1]**2),(len(d),1))
        complex_nums = d.view(dtype = np.complex128)
        amp_phases = np.column_stack([np.abs(complex_nums), np.angle(complex_nums)])
        normalized_data = (uv_data - uv_mean) / uv_std_dev

        uv_distances = np.sqrt(normalized_data[:,0]**2 + normalized_data[:,1]**2).reshape(-1, 1)
        uv_orientation = np.arctan(normalized_data[:,1]/normalized_data[:,0]).reshape(-1, 1)

        # Normalize the array

        data = np.concatenate([normalized_data,amp_phases,uv_distances,uv_orientation],axis=1)

        visibilities.append(data)
    fits_data.append(np.array(fits_file)[0].data)


fits_data = np.array(fits_data)
visibilities_1 = np.array(visibilities)
summary_data_1 = pd.read_csv(f"{data_folder}/Summary.txt",delimiter=" ")

  0%|          | 0/1000 [00:00<?, ?it/s]

TimeoutError: [Errno 60] Operation timed out

In [11]:

dec=-np.pi/4
mas_factor = 4.8481368110954e-9
image_scale_ra  = (np.array([.6]) * mas_factor) / np.cos(dec)
image_scale_ra

array([4.1137805e-09])

In [12]:

dec=-np.pi/4
mas_factor = 4.8481368110954e-9
image_scale_ra  = (np.array([.6]) * mas_factor) / np.cos(dec)
image_scale_ra
mas_factor = 4.8481368110954e-9
image_scale_dec  = (np.array([.6]) * mas_factor)
image_scale_dec

array([2.90888209e-09])

In [ ]:
# Rescale to between 0 and scale value

summary_data_1.ra_0 = (summary_data_1.ra_0 + image_scale_ra) / (2* image_scale_ra)


summary_data_1.dec_0 = ((summary_data_1.dec_0 - dec) + image_scale_dec) / (2* image_scale_dec)

# Different kinds of creating pairings for the network, not sure which one is best

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as hcluster

def create_pairings(d1,d2,thresh=.1):
    X = np.array([d1,d2]).T
# clustering
    y_kmeans = hcluster.fclusterdata(X, thresh, criterion="distance")
    print(len(np.unique(y_kmeans)))

    # Get the predicted cluster labels
    #y_kmeans = clusterskmeans.predict(X)

    pairings = []
    for x in range(y_kmeans.max()+1):
        cluster = np.argwhere(y_kmeans==x).ravel()
        if len(cluster) > 1:
            pairings.extend([[[x,y] for y in cluster if y != x ]for x in cluster][0])

    pairings  = np.array(pairings)
    return pairings

In [15]:
import numpy as np
import scipy.spatial.distance as ssd

def create_pairings3(d1, d2, min_cluster_size=5, max_cluster_size=3000, max_inter_cluster_distance=.05, max_inter_cluster_edges=20):
    """
    Generate edge indices for a graph neural network based on clustering of input data.

    Parameters:
    d1 (numpy.ndarray): First set of input data points.
    d2 (numpy.ndarray): Second set of input data points.
    min_cluster_size (int): Minimum size of a cluster to be considered.
    max_cluster_size (int): Maximum size of a cluster to be considered.
    max_inter_cluster_distance (float): Maximum distance between clusters to add inter-cluster edges.
    max_inter_cluster_edges (int): Maximum number of inter-cluster edges to add.

    Returns:
    numpy.ndarray: Edge indices for the graph.
    """
    X = np.column_stack((d1, d2))

    # Compute distance matrix
    distance_matrix = ssd.squareform(ssd.pdist(X))

    # Perform DBSCAN c  lustering
    from sklearn.cluster import DBSCAN
    dbscan = DBSCAN(eps=max_inter_cluster_distance, min_samples=min_cluster_size)
    labels = dbscan.fit_predict(X)
    unique_labels = np.unique(labels)

    pairings = []

    # Add edges within clusters
    for label in unique_labels:
        if label == -1:  # Ignore noise points
            continue

        cluster = np.where(labels == label)[0]
        if min_cluster_size <= len(cluster) <= max_cluster_size:
            pairings.extend([[[x, y] for y in cluster if y != x] for x in cluster][0])

    # Add edges between clusters
    for i in range(len(unique_labels)):
        for j in range(i + 1, len(unique_labels)):
            if i != j:
                cluster_i = np.where(labels == unique_labels[i])[0]
                cluster_j = np.where(labels == unique_labels[j])[0]

                # Add edges between clusters if the maximum distance is within the threshold
                if np.min(distance_matrix[np.ix_(cluster_i, cluster_j)]) <= max_inter_cluster_distance:
                    for _ in range(max_inter_cluster_edges):
                        node_a = np.random.choice(cluster_i)
                        node_b = np.random.choice(cluster_j)
                        if [node_a, node_b] not in pairings and [node_b, node_a] not in pairings:
                            pairings.append([node_a, node_b])

    return np.array(pairings)

In [16]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_data_list = []
test_data_list = []

def create_data(vis,source_data):
    data_list = []
    for x,y in tqdm(zip(vis,source_data)):
        tensor = torch.tensor(data=x,dtype=torch.float)

        edge_index_np = create_pairings3(x[:, 0], x[:, 1])
        edge_index = torch.tensor(edge_index_np.T, dtype=torch.long)  # Transpose to (2, num_edges) format


        # Compute edge attributes as relative (u, v) differences
        edge_attr = tensor[edge_index[1], :2] - tensor[edge_index[0], :2]  # Difference in u, v coordinates

        ys = torch.tensor([y], dtype=torch.float)  # Target predictions
        time = torch.tensor(np.arange(len(x),step=1))
        # Add edge_attr to the Data object
        data_row = Data(x=tensor, y=ys, edge_index=edge_index,edge_attr=edge_attr).to(device)

        data_list.append(data_row)
    return data_list


In [ ]:

train_data_list_1 = create_data(visibilities_1[:900],summary_data_1.to_numpy()[:900])
test_data_list_1 = create_data(visibilities_1[900:],summary_data_1.to_numpy()[900:])


0it [00:00, ?it/s]

/var/folders/ln/x7tb_x112yn9bhb5qk2197qc0000gp/T/ipykernel_74299/2371927204.py:21: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:281.)
  ys = torch.tensor([y], dtype=torch.float)  # Target predictions


0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [ ]:
import random

all_train_data = []
all_test_data = []

all_train_data.extend(train_data_list_1)
all_test_data.extend(test_data_list_1)

In [20]:

torch.save(all_train_data,'./train_data_saved_3000_observations.pt')

torch.save(all_test_data,'./test_data_saved_3000_observations.pt')